
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Transforming PDFs to Structured Data
## Overview
In this demo, we'll use Databricks AI Functions to transform raw PDFs into structured tabular data. We'll parse documents, classify their content, and extract specific fields, turning unstructured files into queryable tables.

## Learning Objectives
By the end of this demo, you will be able to:
1. Parse a PDF using `ai_parse_document`
2. Classify parsed content using `ai_classify`
3. Extract structured fields using `ai_extract`
4. Store the results in a Delta table

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup

In [0]:
%run ./Includes/Classroom-Setup-04

As a part of the workspace setup, certain assets have been managed for you (volumes, PDF, etc.). Here, we read in from the shared catalog `dbacademy`, schema `ka_all`, and volume `vol_all` the PDF `airbnb_listings.pdf`.

In [0]:
# Variables from the classroom setup
# These reference the shared volume with Airbnb PDFs
SHARED_VOLUME = "/Volumes/dbacademy/ka_all/vol_all"
LISTINGS_PDF = f"{SHARED_VOLUME}/listings/airbnb_listings.pdf"

print(f"Listings PDF:    {LISTINGS_PDF}")

## B. Parsing Documents with `ai_parse_document`

Let's start by parsing the Airbnb listings PDF. The `ai_parse_document` function reads the binary content and returns a structured JSON representation of the document.
- We'll use the pre-defined variable from above `LISTINGS_PDF`
- We will also use `MAP('version', '2.0)`. This pins the parse output to the v2.0 schema (elements array, bbox, descriptions, etc.) instead of relying on whatever the current default version is.

1. After running the next cell, expand the **parsed** column to view the parsed PDF contents.

In [0]:
parsed_df = spark.sql(f"""
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed
  FROM READ_FILES(
    '{LISTINGS_PDF}',
    format => 'binaryFile'
  )
""")

display(parsed_df)

### B1. Exploring the Parsed Output

The parsed output is a single `VARIANT` column containing a nested JSON structure. To work with individual content blocks, we need to flatten it. The query below uses two key SQL patterns:

- **`LATERAL variant_explode(parsed:document:elements)`**: The colon (`:`) operator navigates into the VARIANT to reach the `elements` array. `variant_explode` ([AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/variant_explode) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/variant_explode) | [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/variant_explode)) then converts each array element into its own row, giving us one row per content block. The `LATERAL` keyword lets it reference the `parsed` column from the same `FROM` clause.
- **`element.value:type::STRING`**: Each exploded row has a `value` column (the array element as VARIANT). The `:` operator drills into fields like `type` and `content`, and `::STRING` casts the result to a string.

In [0]:
elements_df = spark.sql(f"""
  WITH parsed_docs AS (
    SELECT
      path,
      ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES(
      '{LISTINGS_PDF}',
      format => 'binaryFile'
    )
  )
  SELECT
    path,
    element.value:type::STRING AS element_type,
    element.value:content::STRING AS content
  FROM
    parsed_docs,
    LATERAL variant_explode(parsed_docs.parsed:document:elements) AS element
  WHERE element.value:content IS NOT NULL
""")

display(elements_df)

## C. Classifying Content with `ai_classify`

Now let's classify each parsed element. We'll categorize them into types like property descriptions, rules, pricing info, etc. A few things to note in the query below:

- **`LENGTH(...) > 50`**: Filters out short elements (headers, page numbers, etc.) that don't carry enough text for meaningful classification.
- **`ai_classify(content, '["property_description", ...]')`**: The second argument is a JSON array of candidate labels. The AI model reads the text and picks the best match.
- **`cls:response[0]`**: `ai_classify` returns a VARIANT with a `response` array. The `[0]` index retrieves the top predicted label.

In [0]:
classified_df = spark.sql(f"""
WITH elements AS (
  SELECT
    element.value:content::STRING AS content
  FROM (
    SELECT ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES('{LISTINGS_PDF}', format => 'binaryFile')
  ) t,
  LATERAL variant_explode(parsed:document:elements) AS element
  WHERE element.value:content IS NOT NULL
    AND LENGTH(element.value:content::STRING) > 50
),
classified AS (
  SELECT
    content,
    ai_classify(
      content,
      '["property_description","pricing_information","location_details","amenities","host_information","other"]'
    ) AS cls
  FROM elements
)
SELECT
  content,
  cls:response[0] AS category
FROM classified
LIMIT 20
""")
display(classified_df)

## D. Extracting Structured Fields with `ai_extract`

Let's extract specific fields from the parsed listing text: property name, location, property type, and amenities. Note:

- **`LENGTH(...) > 100`**: Uses a higher threshold than `ai_classify` because extraction needs enough surrounding text to locate meaningful field values.
- **`ai_extract(content, '["property_name", "neighborhood", ...]')`**: The second argument lists the field names you want. The AI model scans the text and returns a JSON object with those keys populated.
- **`ext:response.property_name`**: The result VARIANT has a `response` object. Use dot notation to access each extracted field by name.

In [0]:
extracted_df = spark.sql(f"""
WITH elements AS (
  SELECT
    element.value:content::STRING AS content
  FROM (
    SELECT ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES('{LISTINGS_PDF}', format => 'binaryFile')
  ) t,
  LATERAL variant_explode(parsed:document:elements) AS element
  WHERE element.value:content IS NOT NULL
    AND LENGTH(element.value:content::STRING) > 100
),
extracted AS (
  SELECT
    content,
    ai_extract(
      content,
      '["property_name", "neighborhood", "property_type", "key_amenities"]'
    ) AS ext
  FROM elements
)
SELECT
  content,
  ext:response.property_name   AS property_name,
  ext:response.neighborhood    AS neighborhood,
  ext:response.property_type   AS property_type,
  ext:response.key_amenities   AS key_amenities
FROM extracted
LIMIT 15
""")

display(extracted_df)

We can also explore the raw `ai_extract` VARIANT output directly, without unpacking individual fields. This lets you see the full JSON structure returned by the function, useful for understanding the schema before deciding which fields to extract.

In [0]:
# Explore the raw ai_extract output
explore_df = spark.sql(f"""
WITH elements AS (
  SELECT
    element.value:content::STRING AS content
  FROM (
    SELECT ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES('{LISTINGS_PDF}', format => 'binaryFile')
  ) t,
  LATERAL variant_explode(parsed:document:elements) AS element
  WHERE element.value:content IS NOT NULL
    AND LENGTH(element.value:content::STRING) > 100
)
SELECT
  content,
  ai_extract(
    content,
    '["property_name", "neighborhood", "property_type", "key_amenities"]'
  ) AS ext
FROM elements
LIMIT 5
""")

display(explore_df)

## E. Storing Results in a Delta Table

Let's combine parsing, classification, and extraction into a single pipeline and save the results. The query below chains everything into one `CREATE OR REPLACE TABLE` statement:

1. The `elements` CTE parses the PDF and flattens it (same as section B1)
2. The `classified_extracted` CTE runs both `ai_classify` and `ai_extract` on each element. Since they're in the same `SELECT`, Databricks can process them in parallel per row
3. The final `SELECT` unpacks both results and writes them as named columns

Run the cell below to create a structured table from the parsed PDF content.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog_name}.{schema_name}.airbnb_structured_listings AS
WITH elements AS (
  SELECT
    element.value:content::STRING AS content
  FROM (
    SELECT ai_parse_document(content, MAP('version', '2.0')) AS parsed
    FROM READ_FILES('{LISTINGS_PDF}', format => 'binaryFile')
  ) t,
  LATERAL variant_explode(parsed:document:elements) AS element
  WHERE element.value:content IS NOT NULL
    AND LENGTH(element.value:content::STRING) > 100
),
classified_extracted AS (
  SELECT
    content,
    ai_classify(
      content,
      '["property_description","pricing_information","location_details","amenities","host_information","other"]'
    ) AS cls,
    ai_extract(
      content,
      '["property_name","neighborhood","property_type","key_amenities"]'
    ) AS ext
  FROM elements
)
SELECT
  content,
  cls:response[0]                 AS category,
  ext:response.property_name      AS property_name,
  ext:response.neighborhood       AS neighborhood,
  ext:response.property_type      AS property_type,
  ext:response.key_amenities      AS key_amenities
FROM classified_extracted
""")

print("Table created: airbnb_structured_listings")

In [0]:
%sql
SELECT * FROM airbnb_structured_listings

## F. Conclusion
In this demo, you:

1. **Parsed** PDFs using `ai_parse_document` to extract structured content
2. **Classified** parsed text into categories using `ai_classify`
3. **Extracted** structured fields using `ai_extract`
4. **Stored** the results in a Delta table for downstream analysis

This **Parse → Classify → Extract** pattern is ideal for turning unstructured documents into structured, queryable data. In the next demo, we'll explore the other pipeline: **Parse → Prep Search → AI Search** for building a retrieval index.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
